# Calibration to the geodetic MB: one number, three parameters

OGGM calibrates the mass balance (MB) model of each glacier against one
number: its average MB over 2000-2020, from geodetic observations (Hugonnet et
al., 2021). The MB model has three parameters (`melt_f`, `prcp_fac`,
`temp_bias`). One equation, three unknowns: infinitely many parameter sets match
the observation exactly. This is called **equifinality**.

`mb_calibration_from_geodetic_mb` deals with this by fixing parameters: by
default, it calibrates `melt_f` and keeps the two others fixed. The "informed
three-step" calibration, used for OGGM's preprocessed directories, starts from
informed guesses of the three parameters and moves them one after another, only
as far as needed.

Here we ask what the geodetic MB tells us *on its own*. We calibrate the three
parameters together with AdaPBS, and a prior which is as uninformative as
possible. We will see that:

- AdaPBS is not useless: all its particles match the observation, and it rules
  out a large part of the parameter space;
- but what remains is a surface which spans almost the whole plausible range of
  each parameter: the posterior of each parameter is close to its prior;
- these parameter sets are equally good, but not equivalent: they disagree on
  the winter MB, and on how sensitive the glacier is to warming;
- more precise data of the same kind do not help. A prior does, or other
  kinds of observations.

This notebook follows `wgms_static_calibration.ipynb`, which explains how
AdaPBS works (prior, forward model, iterations, weights): read it first.

AdaPBS is the work of K. Aalstad and E. Alonso-González (MuSA). If you use it,
please cite Aalstad et al. (2026, https://doi.org/10.5194/gmd-19-8565-2026) and
Alonso-González et al. (2022, https://doi.org/10.5194/gmd-15-9127-2022).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from oggm import cfg, utils, workflow
from oggm.core.massbalance import (mb_calibration_from_geodetic_mb,
                                   decide_winter_precip_factor)

from oggm_adapbs import glogit, gexpit, weighted_summary
from oggm_adapbs.massbalance import (mb_calibration_from_geodetic_mb_adapbs, mb_prior,
                                     static_mb_model, seasonal_specific_mb)

## Glacier and data

We use Vernagtferner (Ötztal Alps, Austria) again, as in the first notebook. Its
WGMS seasonal MB will tell us, at the end, what the geodetic MB cannot. We start
from the OGGM preprocessed directories at level 3 (W5E5 climate, elevation band
flowlines).

In [ ]:
cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = utils.gettempdir('adapbs_geodetic_equifinality', reset=True)

rgi_id = 'RGI60-11.00719'  # Vernagtferner
base_url = ('https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/'
            'L3-L5_files/2025.6/elev_bands/W5E5/per_glacier/')
gdir = workflow.init_glacier_directories([rgi_id], from_prepro_level=3,
                                         prepro_border=80,
                                         prepro_base_url=base_url)[0]

# The temperature bias file used by OGGM's informed calibration
temp_bias_file = ('https://cluster.klima.uni-bremen.de/~oggm/ref_mb_params/oggm_v1.6/'
                  'w5e5_rgi6_perglacier_temp_bias_v2025.6.2.csv')
gdir.name

OGGM's table of geodetic MB (`utils.get_geodetic_mb_dataframe`, from Hugonnet et
al., 2021) has three periods for each glacier: 2000-2010, 2010-2020 and
2000-2020, in m w.e. yr-1, with their errors (1 sigma). We convert them to
kg m-2 yr-1. OGGM uses the 20-year period by default.

In [ ]:
geod = utils.get_geodetic_mb_dataframe().loc[[rgi_id]].set_index('period')
geod = geod[['dmdtda', 'err_dmdtda']] * 1000  # m w.e. yr-1 -> kg m-2 yr-1
print('Default period:', gdir.settings['geodetic_mb_period'])
geod.round(0)

The glacier lost on average 913 kg m-2 per year (913 mm w.e.) between 2000
and 2020, with an error of 154 kg m-2 yr-1. This is the one number we
calibrate against.

# Four deterministic calibrations

`mb_calibration_from_geodetic_mb` calibrates up to three parameters, *one after
the other*: it varies the first one within its bounds until the modelled MB
matches the observation, and moves to the next one only if the first one cannot
do it. The parameters which are not calibrated keep their starting values,
taken from the glacier settings.

The preprocessed directories were already calibrated (with the informed
three-step method): their settings hold the calibrated values. We set the
starting values back to OGGM's defaults first: `melt_f` = 5, `prcp_fac` from
the winter precipitation (`decide_winter_precip_factor`), `temp_bias` = 0.

In [ ]:
params = ['melt_f', 'prcp_fac', 'temp_bias']
print('Calibrated in the preprocessed directory:',
      {p: round(gdir.settings[p], 2) for p in params})

# Back to OGGM's default starting values
gdir.settings['melt_f'] = cfg.PARAMS['melt_f']
gdir.settings['prcp_fac'] = decide_winter_precip_factor(gdir)
gdir.settings['temp_bias'] = 0.
print('Starting values:', {p: round(gdir.settings[p], 2) for p in params})

Now four calls of `mb_calibration_from_geodetic_mb`, which differ only in the
parameter calibrated first (`write_to_gdir=False`: we only want the values):

- `melt_f` only (the default);
- `temp_bias` only;
- `prcp_fac` only;
- the informed three-step calibration (`informed_threestep=True`): `temp_bias`
  is set from the temperature bias file, then `prcp_fac` moves within ±20% of
  its starting value, then `melt_f`, then `temp_bias` if needed.

In [ ]:
calls = {
    'melt_f only': dict(calibrate_param1='melt_f'),
    'temp_bias only': dict(calibrate_param1='temp_bias'),
    'prcp_fac only': dict(calibrate_param1='prcp_fac'),
    'informed three-step': dict(informed_threestep=True,
                                temp_bias_file_path=temp_bias_file),
}
det = {}
for name, kwargs in calls.items():
    d = mb_calibration_from_geodetic_mb(gdir, write_to_gdir=False, **kwargs)
    det[name] = {p: float(d[p]) for p in params}
det = pd.DataFrame(det).T

To check them, we compute the MB of each parameter set ourselves, with the
static MB model of the first notebook (`static_mb_model`: the MB model on the
inversion flowlines). The period 2000-01-01 to 2020-01-01 is the 20 years
2000 to 2019. `set_params` sets the parameters of the MB model.

In [ ]:
years = np.arange(2000, 2020)
mb_mod, fls = static_mb_model(gdir, det.iloc[0].to_dict())

def set_params(theta):
    # Set the parameters of the MB model (in the order of params)
    for p, v in zip(params, theta):
        setattr(mb_mod, p, v)

def geodetic_mb(theta):
    # The mean MB 2000-2019 (kg m-2 yr-1) of one parameter set
    set_params(theta)
    return mb_mod.get_specific_mb(fls=fls, year=years).mean()

det['MB 2000-2019'] = [geodetic_mb(th) for th in det[params].values]
det.round(2)

Four parameter sets, four perfect fits. `melt_f` ranges from 5 to 12.4,
`prcp_fac` from 0.50 to 3.36, `temp_bias` from 0 to 2.26 K. Which one is
right? The geodetic MB cannot tell: the choice is made by the method (which
parameter moves first, and from where), not by the data.

# AdaPBS with a flat prior

## There is no "no prior"

AdaPBS needs a prior: it draws its first particles from it, and the posterior is
the prior times the likelihood. The closest we can get to "no prior" is a
**flat** prior: all values within the bounds are about equally likely.

As in the first notebook, the prior of each parameter is logit-normal: Gaussian
in the normal space, with the bounds at $-\infty$ and $+\infty$. Centred in the
middle of the bounds (0 in the normal space) with a standard deviation of 1.4,
it is close to uniform. For the bounds, we take OGGM's default ones, those of
the deterministic calibration. They are wide: `temp_bias` goes from -15 to
+15 K.

Let's see what this prior looks like. We use `mb_prior` (see the first notebook)
for the bounds only, and draw samples in the normal space:

In [ ]:
# OGGM's default bounds, as used by mb_calibration_from_geodetic_mb
oggm_bounds = {p: (gdir.settings[f'{p}_min'], gdir.settings[f'{p}_max']) for p in params}
prior = mb_prior(gdir, params, prior_bounds=oggm_bounds)
x_min, x_max = prior['x_min'], prior['x_max']

# Samples of the flat prior: centred at 0 in normal space, sd 1.4
flat_sd = 1.4
rng = np.random.default_rng(0)
flat_prior = gexpit(flat_sd * rng.standard_normal((10000, 3)), x_min, x_max)

labels = {'melt_f': 'melt_f (kg m-2 day-1 K-1)', 'prcp_fac': 'prcp_fac',
          'temp_bias': 'temp_bias (K)'}
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for k, (ax, p) in enumerate(zip(axes, params)):
    ax.hist(flat_prior[:, k], bins=np.linspace(x_min[k], x_max[k], 41), density=True,
            color='0.7')
    ax.set_xlabel(labels[p])
    ax.set_yticks([])
axes[0].set_ylabel('density')
plt.tight_layout()

Flat in the middle, falling off near the bounds. This is our "we know
nothing" prior: `melt_f` anywhere between 1.5 and 17, `prcp_fac` between 0.1
and 10, `temp_bias` between -15 and +15 K.

## Running AdaPBS

In the first notebook, we wrote the forward model ourselves. Here, the entity
task `mb_calibration_from_geodetic_mb_adapbs` does everything for us: it reads
the geodetic MB and its error, builds the prior, the forward model (the mean MB
2000-2019 on the inversion flowlines, as above) and runs AdaPBS. Its arguments
for the prior:

- `prior_bounds`: the bounds (by default, those of `mb_prior`);
- `centre_prior=False`: centre the prior in the middle of the bounds, not on
  OGGM's informed values;
- `prior_sd`: the standard deviation in normal space.

And for AdaPBS: we use 1000 particles per iteration, and allow up to 20
iterations. A weakly constrained problem needs more of both: the posterior is a
thin, curved surface, which is hard to approximate with the Gaussian proposals
of AdaPBS. Each model run takes less than a millisecond, so this is cheap.

In [ ]:
adapbs_kwargs = dict(n_particles=1000, max_iterations=20, neff_threshold=0.3, seed=0)

ds = mb_calibration_from_geodetic_mb_adapbs(
    gdir,
    prior_bounds=oggm_bounds,  # OGGM's default bounds
    centre_prior=False,        # centred in the middle of the bounds
    prior_sd=flat_sd,          # close to uniform
    **adapbs_kwargs)
ds

The task returns an xarray Dataset:

- one value per particle (dimension `particle`, all the particles of all the
  iterations): the parameters, their posterior `weight`, the predicted MB
  (`mb`) and the AdaPBS `iteration`;
- the resampled, equally weighted particles (dimension `resampled`): the
  `posterior_*` variables;
- the observation (`ref_mb`) and its error (`ref_mb_err`);
- the AdaPBS diagnostics, one value per iteration (`adapbs_*`);
- the prior and the log evidence (see below) in the attributes.

In [ ]:
ds[['adapbs_n_history', 'adapbs_neff', 'adapbs_neff_frac']].to_dataframe().round(2)

AdaPBS needed 6 iterations (6000 model runs, a few seconds) to reach an effective sample
size above 30% of 1000.

Two small helper functions, to get the particles and their weights out of the
Dataset, and the quantiles of the posterior (`weighted_summary`, see the first
notebook):

In [ ]:
def particles(ds):
    # All the particles: one row per particle, one column per parameter
    return np.stack([ds[p].values for p in params], axis=1)

def resampled(ds):
    # The equally weighted resampled particles
    return np.stack([ds[f'posterior_{p}'].values for p in params], axis=1)

def summary(ds):
    return weighted_summary(particles(ds), ds['weight'].values, params)

def fmt(row):
    return f"{row['q50']:.2f} [{row['q5']:.2f}, {row['q95']:.2f}]"

flat_prior_df = weighted_summary(flat_prior, np.ones(len(flat_prior)) / len(flat_prior), params)
pd.DataFrame({'flat prior': flat_prior_df.apply(fmt, axis=1),
              'posterior': summary(ds).apply(fmt, axis=1)}).T

## The posterior

The table gives the median and 5-95% range of each parameter. The ranges of
`melt_f` and `prcp_fac` barely change: the posterior of these parameters is
almost their prior. Only `temp_bias` is narrower: values far from 0 are ruled
out (with -10 K, the glacier would grow whatever the other parameters are).

The posterior is not exactly the prior either: the medians of `melt_f` and
`prcp_fac` are lower. Where `melt_f` and `prcp_fac` are low, the MB is less
sensitive to temperature, so that a wider range of `temp_bias` values matches
the observation (within its error): these combinations get more weight.

In [ ]:
colors = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']
markers = {'melt_f only': 'o', 'temp_bias only': 's', 'prcp_fac only': '^',
           'informed three-step': 'X'}

def plot_marginals(results, priors=None):
    # Weighted histograms of the particles, one panel per parameter.
    # results: {name: (Dataset, color)}; priors: {name: (samples, color)}
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    for k, (ax, p) in enumerate(zip(axes, params)):
        bins = np.linspace(x_min[k], x_max[k], 31)
        if p == 'temp_bias':
            bins = np.linspace(-6, 10, 31)
        for name, (samples, c) in (priors or {}).items():
            ax.hist(samples[:, k], bins=bins, density=True, histtype='step', ls='--',
                    lw=1.2, color=c, label=name)
        for name, (r, c) in results.items():
            ax.hist(r[p].values, bins=bins, weights=r['weight'].values, density=True,
                    histtype='step', lw=2, color=c, label=name)
        ax.set_xlabel(labels[p])
        ax.set_yticks([])
    axes[0].set_ylabel('density')
    axes[-1].legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
    plt.tight_layout()

plot_marginals({'posterior': (ds, colors[0])}, priors={'flat prior': (flat_prior, '0.5')})

(The `temp_bias` panel is zoomed in on -6 to +10 K: the prior extends from -15
to +15 K.)

The marginals hide the most important thing: how the parameters depend on each
other. The pair plots show the resampled particles, for each pair of
parameters, coloured by the third one. The black markers are the four
deterministic calibrations.

In [ ]:
def plot_pairs(theta):
    pairs = [('melt_f', 'temp_bias', 'prcp_fac'), ('prcp_fac', 'temp_bias', 'melt_f'),
             ('melt_f', 'prcp_fac', 'temp_bias')]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (px, py, pc) in zip(axes, pairs):
        ix, iy, ic = (params.index(p) for p in (px, py, pc))
        sc = ax.scatter(theta[:, ix], theta[:, iy], c=theta[:, ic], s=8, cmap='viridis')
        plt.colorbar(sc, ax=ax, label=labels[pc])
        for name, row in det.iterrows():
            ax.scatter(row[px], row[py], marker=markers[name], s=90, color='k',
                       edgecolor='w', label=name, zorder=5)
        ax.set_xlabel(labels[px])
        ax.set_ylabel(labels[py])
    axes[0].legend(loc='upper right', fontsize=9)
    plt.tight_layout()

plot_pairs(resampled(ds))

This is equifinality. The particles fill a two-dimensional surface in the
three-dimensional parameter space:

- left: for a given `prcp_fac` (a given colour), `melt_f` and `temp_bias`
  trade off along a curve. More melt per degree (higher `melt_f`) needs fewer
  degrees (lower `temp_bias`). A higher `prcp_fac` shifts the curve up: more
  snow needs more melt;
- middle: for a given `melt_f`, more snow (higher `prcp_fac`) needs a higher
  `temp_bias`;
- right: `melt_f` and `prcp_fac` alone tell us nearly nothing: (almost) any
  pair works, with the right `temp_bias`.

The four deterministic calibrations all lie on this surface: each of them is
one point of it, picked by the method.

## The fit

All these particles fit the observation. The histogram shows the predicted MB of
the particles (with their weights), and the observation with its error:

In [ ]:
obs, obs_err = ds['ref_mb'].values[0], ds['ref_mb_err'].values[0]
mb = ds['mb'].values[:, 0]

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.linspace(obs - 5 * obs_err, obs + 5 * obs_err, 31)
ax.hist(mb, bins=bins, weights=ds['weight'].values, density=True, color=colors[0],
        alpha=0.6, label='AdaPBS particles')
x = np.linspace(bins[0], bins[-1], 200)
ax.plot(x, np.exp(-0.5 * ((x - obs) / obs_err) ** 2) / (obs_err * np.sqrt(2 * np.pi)),
        color='k', label='observation ± error')
ax.set_xlabel('MB 2000-2019 (kg m-2 yr-1)')
ax.set_yticks([])
ax.legend(frameon=False)

# The particles of the first iteration are samples of the prior
first = ds['iteration'].values == 1
q = np.percentile(mb[first], [5, 95])
print(f'MB of the prior samples, 5-95%: {q[0]:.0f} to {q[1]:.0f} kg m-2 yr-1')

The particles reproduce the observation and its error: the posterior predictive
distribution of the MB is the observation. Before calibration (the samples of
the prior), the MB ranged from -25 600 to +6 700 kg m-2 yr-1 (5-95%).

So AdaPBS did its job: it found all the parameter sets which agree with the
observation. It is the observation which does not say much about the
parameters.

# How much did we learn? The information gain

How can we measure how much the observation told us? A standard measure is the
**information gain**: the Kullback-Leibler (KL) divergence of the posterior
from the prior,

$$D_{KL} = \int p(\theta \mid \mathrm{obs}) \,
\log \frac{p(\theta \mid \mathrm{obs})}{p(\theta)} \, d\theta .$$

It is 0 when the posterior is the prior (we learned nothing), and grows as the
posterior becomes narrower than the prior. It is in nats (natural logarithm). An
intuition: if prior and posterior were both uniform, the posterior would
occupy a fraction $e^{-D_{KL}}$ of the volume of the prior.

AdaPBS gives us all we need to compute it. With Bayes' theorem,
$p(\theta \mid \mathrm{obs}) / p(\theta) = L(\theta) / Z$, where $L$ is the
likelihood and $Z$ the **evidence** (the probability of the observation under
the prior, the normalising constant of the posterior). So:

$$D_{KL} = \mathrm{E}_{\mathrm{posterior}}[\log L(\theta)] - \log Z .$$

The first term is the weighted mean of the log-likelihood of the particles
(Gaussian, from their predicted MB). AdaPBS estimates $\log Z$ along with the
weights: it is in the attributes of the Dataset (`log_evidence`).

In [ ]:
def information_gain(ds):
    # KL divergence of the posterior from the prior (nats)
    err = ds['ref_mb_err'].values
    resid = (ds['mb'].values - ds['ref_mb'].values) / err  # one row per particle
    log_lik = np.sum(-0.5 * np.log(2 * np.pi * err ** 2) - 0.5 * resid ** 2, axis=1)
    return np.sum(ds['weight'].values * log_lik) - ds.attrs['log_evidence']

kl = information_gain(ds)
print(f'Information gain: {kl:.2f} nats, exp(KL) = {np.exp(kl):.0f}')

The posterior occupies about 1/50 of the prior volume, in the sense above. This
is not nothing: as we saw, most of it comes from ruling out values of
`temp_bias` far from 0, which the flat prior allowed. But it is information
about one *combination* of the parameters (the surface), not about each of
them: the marginals of `melt_f` and `prcp_fac` are almost unchanged.

How large is large? We get a feeling for it in the next section.

# More data of the same kind

What if the geodetic MB was more precise? Or if we used the two decades
2000-2010 and 2010-2020 instead of the 20 years? We run the same calibration:

- with half the observation error (`obs_err_factor=0.5`);
- with the two decadal values as two observations (`ref_mb_period` with two
  periods: the task then has two observations, with their own errors).

In [ ]:
flat = dict(prior_bounds=oggm_bounds, centre_prior=False, prior_sd=flat_sd)
results = {'20 years': ds}
results['20 years, half error'] = mb_calibration_from_geodetic_mb_adapbs(
    gdir, obs_err_factor=0.5, **flat, **adapbs_kwargs)
results['two decades'] = mb_calibration_from_geodetic_mb_adapbs(
    gdir, ref_mb_period=['2000-01-01_2010-01-01', '2010-01-01_2020-01-01'],
    **flat, **adapbs_kwargs)

def table(results):
    rows = {}
    for name, r in results.items():
        row = summary(r).apply(fmt, axis=1).to_dict()
        row['iterations'] = r.sizes['adapbs_iteration']
        row['final Neff/Ne'] = f"{r['adapbs_neff_frac'].values[-1]:.2f}"
        row['KL (nats)'] = f'{information_gain(r):.2f}'
        rows[name] = row
    return pd.DataFrame(rows).T

table(results)

Halving the error adds 0.65 nats of information, about $\log 2 = 0.69$:
exactly what we expect when the observation constrains a single direction of the
parameter space (the surface gets twice as thin), and nothing else. The
marginals do not change: a thinner surface is still a surface. It is only
harder to sample: AdaPBS needs more iterations.

The two decades are two observations, but they constrain the same thing (the MB
of the decades differ by less than their errors, and the model gets the
difference from the climate data anyway), and each of them is less precise
than the 20-year value: no gain either.

More data of the same kind do not help. We need either a prior (next sections)
or observations of another kind.

# Equally good, but not equivalent

Does it matter which parameter set we pick, if they all fit the observation?
For the past 20 years, not much. But the parameter sets describe different
glaciers. We look at two quantities, for each resampled particle:

- the winter MB (October to April) of the hydrological years 2000-2019, which
  WGMS measured on this glacier (`seasonal_specific_mb`, see the first
  notebook);
- the sensitivity to warming: the change of the mean MB 2000-2019 when the
  temperature is 1 K higher (same geometry).

In [ ]:
def equifinal_check(theta):
    # The geodetic MB, the mean winter MB and the MB change for +1 K of each
    # parameter set (one row each)
    rows = []
    for th in theta:
        mb0 = geodetic_mb(th)
        winter = seasonal_specific_mb(mb_mod, fls, years)['winter'].mean()
        mb1 = geodetic_mb(th + [0, 0, 1])  # 1 K warmer
        rows.append({'MB 2000-2019': mb0, 'winter MB': winter, 'dMB / +1 K': mb1 - mb0})
    return pd.DataFrame(rows)

check = equifinal_check(resampled(ds))
check_det = equifinal_check(det[params].values).set_index(det.index)

# The WGMS winter MB of the same years
wgms = gdir.get_ref_mb_data(y0=2000, y1=2019)
wgms_winter = wgms['WINTER_BALANCE'].mean()
print(f'WGMS winter MB 2000-2019: {wgms_winter:.0f} kg m-2 '
      f"({wgms['WINTER_BALANCE'].count()} years)")

summ = check.quantile([0.05, 0.5, 0.95]).T
summ.columns = ['q5', 'q50', 'q95']
pd.concat([summ, check_det.T], axis=1).round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
th = resampled(ds)
sc = ax.scatter(check['winter MB'], check['dMB / +1 K'], c=th[:, 1], s=8, cmap='viridis')
plt.colorbar(sc, ax=ax, label=labels['prcp_fac'])
for name, row in check_det.iterrows():
    ax.scatter(row['winter MB'], row['dMB / +1 K'], marker=markers[name], s=90,
               color='k', edgecolor='w', label=name, zorder=5)
ax.axvline(wgms_winter, color='k', ls='--', lw=1.5, label='WGMS winter MB')
ax.set_xlabel('mean winter MB 2000-2019 (kg m-2)')
ax.set_ylabel('MB change for +1 K (kg m-2 yr-1)')
ax.legend(loc='lower left', fontsize=9);

All these particles have the same MB 2000-2019 (within the observation error),
but:

- their winter MB ranges from 242 to 2897 kg m-2 (5-95%). It is
  mostly set by `prcp_fac` (the colours). WGMS measured 874 kg m-2;
- their MB change for 1 K of warming ranges from -539 to -2066
  kg m-2 yr-1: a factor of about 4. Glaciers with a lot of
  accumulation (and therefore a lot of melt, to get the same MB) react more
  strongly to warming.

The sensitivity to warming is what matters for projections: the same
calibration data, with a different choice on the equifinal surface, give
different futures. The deterministic calibrations each pick one point (their
sensitivities range from -572 to -1552 kg m-2 yr-1), and say nothing about
the others. AdaPBS shows the whole range.

Here, the informed three-step calibration does well: its winter MB (935
kg m-2) is close to the WGMS value. Its `prcp_fac` comes from the winter
precipitation, which is precisely the information the geodetic MB lacks.

The WGMS winter MB would select a band of this figure: the calibration with
the winter and summer MB in the first notebook did exactly that. Without such
observations, only the prior can choose.

# The prior decides

When the observations do not constrain the parameters, the posterior is
whatever the prior makes of the equifinal surface. We compare three priors:

- **flat**: as above;
- **informed, OGGM bounds**: centred on OGGM's informed values (`melt_f` = 5,
  `prcp_fac` from the winter precipitation, `temp_bias` from the temperature
  bias file) with a standard deviation of 1, and OGGM's bounds. This is the
  prior of the first notebook;
- **informed, narrow bounds**: the same centres, with the narrower default
  bounds of `mb_prior` (`prcp_fac` within a factor of 2 of its centre,
  `temp_bias` ±3 K). This is the default of
  `mb_calibration_from_geodetic_mb_adapbs`.

In [ ]:
priors = {'flat': results['20 years']}
priors['informed, OGGM bounds'] = mb_calibration_from_geodetic_mb_adapbs(
    gdir, prior_bounds=oggm_bounds, temp_bias_file_path=temp_bias_file, **adapbs_kwargs)
priors['informed, narrow bounds'] = mb_calibration_from_geodetic_mb_adapbs(
    gdir, temp_bias_file_path=temp_bias_file, **adapbs_kwargs)

tab = table(priors)
for name, r in priors.items():
    q = equifinal_check(resampled(r))['dMB / +1 K'].quantile([0.05, 0.95])
    tab.loc[name, 'dMB / +1 K'] = f'[{q[0.05]:.0f}, {q[0.95]:.0f}]'
    tab.loc[name, 'log evidence'] = f"{r.attrs['log_evidence']:.2f}"
tab.drop(columns=['iterations', 'final Neff/Ne'])

To plot the priors, we draw samples of the two informed ones, as for the flat
prior. The prior of each run is stored in the attributes of its Dataset
(centre, bounds and standard deviation, in the parameter space for the centre
and bounds).

In [ ]:
def prior_samples(r, n=10000):
    # Samples of the prior of an AdaPBS run, from the attributes of its Dataset
    lo, hi = r.attrs['prior_min'], r.attrs['prior_max']
    mean = 0.
    if r.attrs['centre_prior']:
        mean = glogit(r.attrs['prior_centre'], lo, hi)
    z = mean + r.attrs['prior_sd'] * np.random.default_rng(0).standard_normal((n, 3))
    return gexpit(z, lo, hi)

plot_marginals({f'{name} (posterior)': (r, c) for (name, r), c in zip(priors.items(), colors)},
               priors={f'{name} (prior)': (prior_samples(r), c)
                       for (name, r), c in zip(priors.items(), colors)})

Dashed: the priors, solid: the posteriors. Each posterior follows its prior,
except for the direction which the geodetic MB constrains. With the informed
priors, the information gain is smaller (there was less to learn: the prior
was already closer to the data), and the spread of the sensitivity to warming
narrows: from [-2066, -539] kg m-2 yr-1 with the flat prior to [-1440, -633]
with the narrow informed prior.

The **log evidence** says how well each prior predicted the observation, before
seeing it. It increases from the flat prior to the narrow informed one: the
informed priors put more of their mass where the observation is (the
difference between the flat and the narrow prior is a factor of
$e^{1.95} \approx 7$). This is a weak argument for them, with one
observation. It does not tell whether the informed values are right: here, the
WGMS winter MB agrees with them, but this need not be the case on other
glaciers, and the narrow prior excludes parts of the equifinal surface which
are physically possible.

# Take-aways

- One geodetic MB value cannot constrain three parameters: the parameter sets
  which fit it form a surface in the parameter space. OGGM's deterministic
  calibrations each pick one point on it; which one depends on the method, not
  on the data.
- With a flat prior, AdaPBS finds this surface: all its particles fit the
  observation, and the posterior of each parameter is close to its prior,
  except for the values ruled out by the observation (`temp_bias` far from 0).
- The information gain (KL divergence) quantifies this: about 3.9 nats. More
  precise data of the same kind add little: halving the error adds about
  $\log 2$ nats, and two decadal values instead of one 20-year value add
  nothing.
- The equifinal parameter sets are not equivalent: their winter MB differs by more
  than a factor of 10 and their sensitivity to warming by a factor of about 4. This uncertainty goes straight into projections.
- With the geodetic MB alone, the prior decides. Choose it with care, report it,
  and check what it does to the quantities you care about. Observations of
  another kind (here the winter MB) are what really constrain the parameters.

## Caveats

- The observation error is that of the geodetic data only: the model error is
  not included (in the first notebook, it dominated). With a larger error, the
  surface would be thicker, but the conclusions would not change.
- "Flat" depends on the parametrisation: a prior flat in `melt_f` is not flat
  in `log(melt_f)`. There is no prior without information.
- The geometry is fixed: this calibrates the MB model, not the glacier
  evolution.